In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    pairwise_distances_argmin,
    classification_report,
    accuracy_score
)

In [ ]:
# Load datasets
train = pd.read_csv("../training_image_features.csv")
test = pd.read_csv("../testing_image_features.csv")

TARGET = "Condition_Class"
CLASSES = ["Excellent", "Good", "Fair", "Poor"]

# Use numeric image features; exclude target-related scores
exclude = {
    "Condition_Score",
    "Damage_Score",
    "Alternative_Use_Suitability"
}

In [ ]:
features = [
    col for col in train.select_dtypes(include=np.number).columns
    if col not in exclude
]

X_train = train[features].replace([np.inf, -np.inf], np.nan)
X_test = test[features].replace([np.inf, -np.inf], np.nan)

# Preprocessing
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train))
X_test = scaler.transform(imputer.transform(X_test))

# Fit hierarchical clustering on a training subset
np.random.seed(42)
sample_size = min(2500, len(X_train))
indices = np.random.choice(
    len(X_train), sample_size, replace=False
)

model = AgglomerativeClustering(
    n_clusters=4,
    linkage="ward"
)

cluster_labels = model.fit_predict(X_train[indices])


In [ ]:
# Calculate cluster centroids
centroids = np.vstack([
    X_train[indices][cluster_labels == cluster].mean(axis=0)
    for cluster in range(4)
])

# Assign training and testing samples to nearest centroid
train_clusters = pairwise_distances_argmin(
    X_train, centroids
)

test_clusters = pairwise_distances_argmin(
    X_test, centroids
)

In [ ]:
# Map clusters to majority condition class
cluster_mapping = {}

for cluster in range(4):
    labels = train.loc[
        train_clusters == cluster, TARGET
    ]

    if len(labels) > 0:
        cluster_mapping[cluster] = labels.mode().iloc[0]
    else:
        cluster_mapping[cluster] = train[TARGET].mode().iloc[0]

# Predict test classes
y_pred = np.array([
    cluster_mapping[c] for c in test_clusters
])

y_test = test[TARGET].astype(str)

# Classification metrics
print("Hierarchical Clustering")
print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(
    y_test,
    y_pred,
    labels=CLASSES,
    zero_division=0
))